In [1]:
import pandas as pd
import os

# New dataset (Wholesale customers data)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(url)

print(df.head())
print("Total rows:", len(df))

# Create the directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save data
df.to_csv("data/raw_sales.csv", index=False)

   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185
Total rows: 440


In [ ]:
import pandas as pd

# Load data
df = pd.read_csv("data/raw_sales.csv")

print("Before Cleaning:")
print(df.shape)
print(df.isnull().sum())

# Handle missing values (using forward fill as a simple method)
df = df.ffill()

# Remove duplicates
df = df.drop_duplicates()

print("\nAfter Cleaning:")
print(df.shape)

# Save cleaned data
df.to_csv("data/cleaned_sales.csv", index=False)

print("\nCleaned Data Sample:")
print(df.head())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/cleaned_sales.csv")

print(df.head())
print(df.describe())

# 1. Correlation Matrix Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.drop(['Channel', 'Region'], axis=1).corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Wholesale Sales Categories')
plt.show()

# Define sales features for plotting
sales_features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

# 2. Histograms for each sales feature
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribution of Wholesale Customer Sales Features', fontsize=16)

for i, feature in enumerate(sales_features):
    row = i // 3
    col = i % 3
    sns.histplot(df[feature], kde=True, ax=axes[row, col])
    axes[row, col].set_title(f'{feature} Distribution')
    axes[row, col].set_xlabel('')
    axes[row, col].set_ylabel('')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent overlap
plt.show()

# 3. Box plots for each sales feature by Channel
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Wholesale Sales Feature Distribution by Channel', fontsize=16)

for i, feature in enumerate(sales_features):
    row = i // 3
    col = i % 3
    sns.boxplot(x='Channel', y=feature, data=df, ax=axes[row, col])
    axes[row, col].set_title(f'{feature} by Channel')
    axes[row, col].set_xlabel('')
    axes[row, col].set_ylabel('')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent overlap
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load the cleaned data again to ensure we have the latest version
df = pd.read_csv("data/cleaned_sales.csv")

# 4. Count Plots for Channel and Region
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Distribution of Customers by Channel and Region', fontsize=16)

sns.countplot(x='Channel', data=df, ax=axes[0], palette='viridis')
axes[0].set_title('Customer Distribution by Channel')
axes[0].set_xlabel('Channel')
axes[0].set_ylabel('Number of Customers')

sns.countplot(x='Region', data=df, ax=axes[1], palette='plasma')
axes[1].set_title('Customer Distribution by Region')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Number of Customers')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# 5. Box plots for each sales feature by Region
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Wholesale Sales Feature Distribution by Region', fontsize=16)

sales_features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

for i, feature in enumerate(sales_features):
    row = i // 3
    col = i % 3
    sns.boxplot(x='Region', y=feature, data=df, ax=axes[row, col])
    axes[row, col].set_title(f'{feature} by Region')
    axes[row, col].set_xlabel('')
    axes[row, col].set_ylabel('')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# 6. Scatter plot between highly correlated features (e.g., Grocery and Detergents_Paper)
plt.figure(figsize=(10, 8))
sns.scatterplot(x='Grocery', y='Detergents_Paper', hue='Channel', data=df, alpha=0.7)
plt.title('Grocery vs. Detergents_Paper Spending, Colored by Channel')
plt.xlabel('Grocery Spending')
plt.ylabel('Detergents_Paper Spending')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/cleaned_sales.csv")

# Select features for clustering (sales categories)
sales_features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X = df[sales_features]

# It's good practice to scale the data before KMeans clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10) # Set n_init explicitly
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(df.head())

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pickle

# Load data
df = pd.read_csv("data/cleaned_sales.csv")

# Define features (sales categories) and target (Channel)
sales_features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X = df[sales_features]
y = df['Channel']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42) # Added random_state for reproducibility
model.fit(X_train, y_train)

# Accuracy
accuracy = model.score(X_test, y_test)
print(f"Model Accuracy for predicting Channel: {accuracy:.4f}")

# Create 'models' directory if it doesn't exist
import os
os.makedirs('models', exist_ok=True)

# Save model
pickle.dump(model, open("models/channel_prediction_model.pkl", "wb"))
print("Model saved to models/channel_prediction_model.pkl")